<a href="https://colab.research.google.com/github/NaveenHerath/llm-from-scratch/blob/main/day8_tiny_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi Headed Attention

In [2]:
#1.imports and settings
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

batch_size = 32       # B: chunks per batch
block_size = 8        # T: the longest context the model can see
n_embd = 32           # C: numbers describing each character
n_head = 4            # new: how many heads run side by side
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3

In [3]:
#2. the data loader and batch loader
!wget -q -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [4]:
#3. measuring the loss
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [5]:
#4. One head
class Head(nn.Module):
    """ one head of masked self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)                                       # (B, T, hs)
        q = self.query(x)                                     # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5     # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)                                     # (B, T, hs)
        return wei @ v                                        # (B, T, hs)

In [6]:
#5. multi headed attention
class MultiHeadAttention(nn.Module):
    """ several heads of self-attention side by side """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, num_heads * head_size)
        out = self.proj(out)                                  # (B, T, n_embd)
        return out

In [7]:
#6. the model- caleed tinygpt
class TinyGPT(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)   # 4 heads of size 8
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                   # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T))    # (T, C)
        x = tok_emb + pos_emb                                       # (B, T, C)
        x = self.sa_heads(x)                                        # (B, T, C)
        logits = self.lm_head(x)                                    # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [8]:
#7. the training loop
def train(model, verbose=True):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = []
    for step in range(max_iters + 1):
        if step % eval_interval == 0:
            l = estimate_loss(model)
            history.append((step, l['train'], l['val']))
            if verbose:
                print(f"step {step:4d}: train {l['train']:.4f}, val {l['val']:.4f}")
        if step == max_iters:
            break
        xb, yb = get_batch('train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return history

In [9]:
#8. train
torch.manual_seed(1337)
model = TinyGPT()
print(sum(p.numel() for p in model.parameters()), 'parameters')
history = train(model)

8609 parameters
step    0: train 4.2040, val 4.2046
step  500: train 2.6131, val 2.6227
step 1000: train 2.4690, val 2.4672
step 1500: train 2.4106, val 2.4076
step 2000: train 2.3514, val 2.3687
step 2500: train 2.3237, val 2.3487
step 3000: train 2.3162, val 2.3309
step 3500: train 2.2796, val 2.3076
step 4000: train 2.2742, val 2.2825
step 4500: train 2.2634, val 2.2774
step 5000: train 2.2550, val 2.2840


In [10]:
#9. generate
torch.manual_seed(42)
start = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(start, max_new_tokens=300)[0].tolist()))


Listloulf a days nath my cone no, oveatn youm fe thoue tar heed lamsto cofrd crakintinor kungnl hee,
To to tindu
Le---wh famend.
CUTAat not lulenct, arebough-reng ind prent not hey buth thunir his, buccoth cod thee hut anoth;
Mofe life turd
Therit me dife thoual wer
ses
To to wou may I thour an-s br


In [11]:
results = {}
for heads in [1, 2, 4, 8]:
    n_head = heads
    torch.manual_seed(1337)
    m = TinyGPT()
    h = train(m, verbose=False)
    results[heads] = round(h[-1][2], 4)
    print(f'{heads} heads of size {n_embd // heads}: val {results[heads]}')
n_head = 4

1 heads of size 32: val 2.428
2 heads of size 16: val 2.3318
4 heads of size 8: val 2.284
8 heads of size 4: val 2.2845
